In [3]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.documents import Document
from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
import os

## Why Contextual Compression?

With short, single-topic documents, a base retriever works fine.  
But real-world documents are **long paragraphs** mixing many topics.  
When you query *"How to stay healthy?"*, the base retriever returns the **entire paragraph** — including unrelated sentences about space, finance, or programming.  

**Contextual Compression** wraps the base retriever with an LLM compressor that **extracts only the relevant part** from each returned document.  
Less noise → better LLM answers.

> Key design: each document below contains a health tip **buried inside unrelated content** to make the problem visible.

In [4]:
documents = [
    Document(page_content=(
        "The Amazon rainforest covers over 5.5 million square kilometres and is home to roughly 10 percent of all species on Earth. "
        "Drinking at least 8 glasses of water a day keeps your body hydrated and supports kidney and organ function. "
        "The river basin produces about 20 percent of the world's fresh-water discharge into the oceans."
    )),
    Document(page_content=(
        "Global stock markets hit record highs in early 2024 driven largely by AI sector growth. "
        "Regular exercise such as 30 minutes of walking daily significantly reduces the risk of heart disease and improves mood and energy levels. "
        "Inflation rates in the US dropped to around 3 percent by the end of 2023."
    )),
    Document(page_content=(
        "Jupiter is the largest planet in our solar system with a mass greater than all other planets combined. "
        "Eating a balanced diet rich in fruits, vegetables, whole grains, and lean proteins is one of the most effective ways to promote long-term health. "
        "Jupiter has at least 95 known moons, including Europa which may harbour a subsurface ocean."
    )),
    Document(page_content=(
        "Python was created by Guido van Rossum and first released in 1991 as a general-purpose scripting language. "
        "Getting 7 to 9 hours of quality sleep each night is essential for mental clarity, physical recovery, and hormonal balance. "
        "Python is now the most popular programming language in data science, AI, and automation."
    )),
    Document(page_content=(
        "The Eiffel Tower was completed in 1889 as the entrance arch for the World's Fair and stands 330 metres tall. "
        "Managing stress through practices like meditation, deep breathing, or yoga has been scientifically shown to reduce cortisol levels and improve overall well-being. "
        "The tower receives about 7 million visitors per year, making it the most visited paid monument in the world."
    )),
]

In [5]:
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_store = FAISS.from_documents(
    documents=documents,
    embedding=embedding_model
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2060.04it/s]


In [6]:
base_retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-72B-Instruct",
    task="text-generation",
    huggingfacehub_api_token=os.environ["HF_TOKEN"],
    max_new_tokens=256,
    temperature=0.1
)

chat_model = ChatHuggingFace(llm=llm)

compressor = LLMChainExtractor.from_llm(chat_model)

compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever
)

In [ ]:
query = "How to stay healthy?"

base_results       = base_retriever.invoke(query)
compressed_results = compression_retriever.invoke(query)

In [ ]:
print("=" * 70)
print("BASE RETRIEVER  — full paragraphs (noisy)")
print("=" * 70)
for i, doc in enumerate(base_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("\n" + "=" * 70)
print("CONTEXTUAL COMPRESSION  — only the relevant sentence")
print("=" * 70)
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

## What the output shows

| | Base Retriever | Contextual Compression |
|---|---|---|
| **What is returned** | Full 3-sentence paragraph | Only the 1 relevant sentence |
| **Noise** | High — Amazon basin stats, stock prices, planet facts | Zero |
| **Token cost to LLM** | ~3× more tokens | Minimal |
| **Answer quality** | LLM has to filter noise itself | Clean, focused context |

**Rule of thumb:** use `ContextualCompressionRetriever` whenever your source documents are long or cover multiple topics — i.e., almost every real-world RAG pipeline.